In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [58]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost, lightgbm, catboost

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import config as cnf 
import utils as utl

### Вывод

In [59]:
df_train = pd.read_csv(cnf.path['train_path'])

# Обработка данных
df_train_priz = df_train[['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']].copy()
df_train_priz['Age'] = df_train_priz['Age'].fillna(df_train_priz['Age'].median())
   
X = df_train_priz.drop('Survived', axis=1)
y = df_train_priz['Survived']

# Подготовка параметров
combo_param = utl.parametrs(cnf.RandomForestParam) + utl.parametrs(cnf.LogRegParam)
combo_param += utl.parametrs(cnf.KNNParam)
combo_param += utl.parametrs(cnf.DecisionTreeParam)
combo_param += utl.parametrs(cnf.CatBoostParam)


# Подготовка кросс валидации

results = []
SK_Fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for combo in combo_param:
    
    params = combo.copy()
    name_model = params.pop('name_model')
    scaler = params.pop('scaler')
    
    array_train_accuracy = []
    array_test_accuracy = []
    
    model = cnf.models[name_model](**params)
    print("Model: ", name_model, "Params: ", params)
    
    for i, (train_index, test_index) in enumerate(SK_Fold.split(X, y)):

        current_X_train = X.iloc[train_index]
        current_y_train = y.iloc[train_index]
        current_X_test  = X.iloc[test_index]
        current_y_test  = y.iloc[test_index]

        if scaler:
            scaler = StandardScaler()
            scaler.fit(current_X_train)
            current_X_train = scaler.transform(current_X_train)
            current_X_test = scaler.transform(current_X_test)
            
        model_clone = clone(model)
        model_clone.fit(current_X_train, current_y_train)
        
        train_accuracy = model_clone.score(current_X_train, current_y_train)
        test_accuracy = model_clone.score(current_X_test, current_y_test)

        array_train_accuracy.append(train_accuracy)
        array_test_accuracy.append(test_accuracy)

    mean_train_accuracy = float(np.mean(array_train_accuracy))
    mean_train_std = float(np.std(array_train_accuracy))
    mean_test_accuracy = float(np.mean(array_test_accuracy))
    mean_test_std = float(np.std(array_test_accuracy))
    
    results.append({
        **combo
        , 'train_accuracy'  :   mean_train_accuracy
        , 'train_std'       :   mean_train_std
        , 'test_accuracy'   :   mean_test_accuracy
        , 'test_std'        :   mean_test_std
    })

Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 42}
Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 100, 'random_state': 42}
Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 3, 'n_estimators': 100, 'random_state': 42}
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 42}
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 2, 'n_estimators': 100, 'random_state': 42}
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 3, 'n_estimators': 100, 'random_state': 42}
Model:  LogReg Params:  {'random_state': 42}
Model:  KNN Params:  {'n_neighbors': 1}
Model:  KNN Params:  {'n_neighbors': 2}
Model:  KNN Params:  {'n_neighbors': 3}
Model:  KNN Params:  {'n_neighbors': 4}
Model:  KNN Params:  {'n_neighbors': 5}
Model:  KNN Params:  {'n_neighbors': 6}
Model:  KNN Params:  {'n_neighbors': 7}
Model:  K

In [60]:
results = pd.DataFrame(results)
results['gap'] = results['train_accuracy'] - results['test_accuracy']

# Порядок колонок: сначала модель, потом метрики, потом гиперпараметры
metrics = ['test_accuracy', 'test_std', 'train_accuracy', 'train_std', 'gap']
service = ['name_model', 'scaler', 'random_state']
params = [c for c in results.columns if c not in metrics + service]

results = results[['name_model'] + metrics + params + ['scaler', 'random_state']]

# Сортировка: лучшие по test_accuracy сверху
results = results.sort_values('test_accuracy', ascending=False).reset_index(drop=True)
results.head(50)

,name_model,test_accuracy,test_std,train_accuracy,train_std,gap,max_depth,min_samples_leaf,n_estimators,n_neighbors,depth,iterations,scaler,random_state
0,RandomForest,0.732867,0.024171,0.801905,0.005939,0.069038,8.0,3.0,100.0,NaN,NaN,NaN,False,42.0
1,CatBoost,0.731756,0.038373,0.753088,0.006386,0.021333,NaN,NaN,NaN,NaN,5.0,20.0,False,42.0
2,RandomForest,0.729477,0.028174,0.834453,0.007155,0.104976,9.0,2.0,100.0,NaN,NaN,NaN,False,42.0
3,CatBoost,0.728366,0.025050,0.781142,0.005351,0.052776,NaN,NaN,NaN,NaN,10.0,20.0,False,42.0
4,RandomForest,0.728366,0.023490,0.812567,0.008971,0.084201,8.0,2.0,100.0,NaN,NaN,NaN,False,42.0
5,CatBoost,0.727268,0.033855,0.748315,0.006052,0.021047,NaN,NaN,NaN,NaN,5.0,10.0,False,42.0
6,RandomForest,0.727255,0.013434,0.840344,0.010275,0.113089,8.0,1.0,100.0,NaN,NaN,NaN,False,42.0
7,RandomForest,0.727236,0.029508,0.810040,0.009758,0.082804,9.0,3.0,100.0,NaN,NaN,NaN,False,42.0
8,CatBoost,0.723903,0.025741,0.770760,0.010771,0.046857,NaN,NaN,NaN,NaN,10.0,10.0,False,42.0
9,RandomForest,0.718285,0.024035,0.870651,0.004726,0.152366,9.0,1.0,100.0,NaN,NaN,NaN,False,42.0
